# 🏀 March Machine Learning Mania 2026 — Modeling Notebook

This notebook turns EDA insights into a full prediction workflow for the NCAA competition.

**Goal:** Predict $P(\text{lower TeamID wins})$ for all required matchups.

**Primary metric:** Brier Score (MSE on predicted probabilities).

---

## Notebook Plan
1. Load and validate competition datasets
2. Engineer team-season features (basic + advanced + schedule + momentum + rankings)
3. Build leakage-safe matchup features
4. Train calibrated probability models with season-aware validation
5. Blend best models and generate final submission

In [ ]:
# ============================================================
# 1. Imports & Global Config
# ============================================================
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, log_loss

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid')

DATA_DIR = Path('/kaggle/input/competitions/march-machine-learning-mania-2026')
TARGET_SEASON = 2026
VALID_SEASONS = [2021, 2022, 2023, 2024, 2025]  # aligned with public LB period
RANDOM_STATE = 42

print('✅ Imports loaded')
print(f'Using data directory: {DATA_DIR}')

In [ ]:
# ============================================================
# 2. Data Loading
# ============================================================
def load_data(data_dir: Path):
    d = {}
    # Core results
    d['m_reg'] = pd.read_csv(data_dir / 'MRegularSeasonCompactResults.csv')
    d['w_reg'] = pd.read_csv(data_dir / 'WRegularSeasonCompactResults.csv')
    d['m_reg_det'] = pd.read_csv(data_dir / 'MRegularSeasonDetailedResults.csv')
    d['w_reg_det'] = pd.read_csv(data_dir / 'WRegularSeasonDetailedResults.csv')
    d['m_tourney'] = pd.read_csv(data_dir / 'MNCAATourneyCompactResults.csv')
    d['w_tourney'] = pd.read_csv(data_dir / 'WNCAATourneyCompactResults.csv')

    # Metadata / supplements
    d['m_seeds'] = pd.read_csv(data_dir / 'MNCAATourneySeeds.csv')
    d['w_seeds'] = pd.read_csv(data_dir / 'WNCAATourneySeeds.csv')
    d['m_team_conf'] = pd.read_csv(data_dir / 'MTeamConferences.csv')
    d['w_team_conf'] = pd.read_csv(data_dir / 'WTeamConferences.csv')
    d['massey'] = pd.read_csv(data_dir / 'MMasseyOrdinals.csv')

    # Submission templates
    d['sub_stage2'] = pd.read_csv(data_dir / 'SampleSubmissionStage2.csv')

    return d

data = load_data(DATA_DIR)
for k, v in data.items():
    print(f'{k:12s} -> {v.shape}')

## 3. Feature Engineering Functions

The feature set is built to reflect tournament performance drivers discovered in EDA:
- Team quality and consistency (win rate, margins)
- Efficiency profile (offense/defense, Four Factors style metrics)
- Opponent quality (strength of schedule)
- Recent form (last-10 trend)
- Tournament priors (seed, rankings)

In [ ]:
# ============================================================
# 3.1 Utility Parsers
# ============================================================
def parse_seed_num(seed: pd.Series) -> pd.Series:
    return seed.astype(str).str[1:3].astype(int)

def parse_submission_ids(sub_df: pd.DataFrame) -> pd.DataFrame:
    out = sub_df.copy()
    parts = out['ID'].str.split('_', expand=True)
    out['Season'] = parts[0].astype(int)
    out['Team1'] = parts[1].astype(int)
    out['Team2'] = parts[2].astype(int)
    out['Gender'] = np.where(out['Team1'] < 2000, 'M', 'W')
    return out

def safe_div(num, den):
    return num / np.where(den == 0, np.nan, den)

In [ ]:
# ============================================================
# 3.2 Team-Season Features from Compact Results
# ============================================================
def build_basic_team_features(reg_compact: pd.DataFrame) -> pd.DataFrame:
    wins = reg_compact.groupby(['Season', 'WTeamID']).agg(
        Wins=('WScore', 'count'),
        WPoints=('WScore', 'sum'),
        WPointsAllowed=('LScore', 'sum'),
    ).reset_index().rename(columns={'WTeamID': 'TeamID'})

    losses = reg_compact.groupby(['Season', 'LTeamID']).agg(
        Losses=('LScore', 'count'),
        LPoints=('LScore', 'sum'),
        LPointsAllowed=('WScore', 'sum'),
    ).reset_index().rename(columns={'LTeamID': 'TeamID'})

    base = wins.merge(losses, on=['Season', 'TeamID'], how='outer').fillna(0)
    base['Games'] = base['Wins'] + base['Losses']
    base['WinPct'] = safe_div(base['Wins'], base['Games'])
    base['PointsFor'] = base['WPoints'] + base['LPoints']
    base['PointsAllowed'] = base['WPointsAllowed'] + base['LPointsAllowed']
    base['PPG'] = safe_div(base['PointsFor'], base['Games'])
    base['PAPG'] = safe_div(base['PointsAllowed'], base['Games'])
    base['AvgMargin'] = base['PPG'] - base['PAPG']

    keep = ['Season', 'TeamID', 'Games', 'Wins', 'Losses', 'WinPct', 'PPG', 'PAPG', 'AvgMargin']
    return base[keep]

def build_recent_form(reg_compact: pd.DataFrame, n_games: int = 10) -> pd.DataFrame:
    w = reg_compact[['Season', 'DayNum', 'WTeamID', 'WScore', 'LScore']].rename(
        columns={'WTeamID': 'TeamID', 'WScore': 'PtsFor', 'LScore': 'PtsAga'}
    )
    w['Win'] = 1

    l = reg_compact[['Season', 'DayNum', 'LTeamID', 'LScore', 'WScore']].rename(
        columns={'LTeamID': 'TeamID', 'LScore': 'PtsFor', 'WScore': 'PtsAga'}
    )
    l['Win'] = 0

    all_games = pd.concat([w, l], ignore_index=True).sort_values(['Season', 'TeamID', 'DayNum'])
    last_n = all_games.groupby(['Season', 'TeamID']).tail(n_games)

    out = last_n.groupby(['Season', 'TeamID']).agg(
        LastNWinPct=('Win', 'mean'),
        LastNPPG=('PtsFor', 'mean'),
        LastNPAPG=('PtsAga', 'mean')
    ).reset_index()
    out['LastNMargin'] = out['LastNPPG'] - out['LastNPAPG']
    return out

def build_strength_of_schedule(reg_compact: pd.DataFrame, basic: pd.DataFrame) -> pd.DataFrame:
    w_opps = reg_compact[['Season', 'WTeamID', 'LTeamID']].rename(columns={'WTeamID': 'TeamID', 'LTeamID': 'OppID'})
    l_opps = reg_compact[['Season', 'LTeamID', 'WTeamID']].rename(columns={'LTeamID': 'TeamID', 'WTeamID': 'OppID'})
    opps = pd.concat([w_opps, l_opps], ignore_index=True)

    opp_wp = basic[['Season', 'TeamID', 'WinPct']].rename(columns={'TeamID': 'OppID', 'WinPct': 'OppWinPct'})
    opps = opps.merge(opp_wp, on=['Season', 'OppID'], how='left')

    sos = opps.groupby(['Season', 'TeamID'])['OppWinPct'].mean().reset_index()
    sos.rename(columns={'OppWinPct': 'SOS'}, inplace=True)
    return sos

In [ ]:
# ============================================================
# 3.3 Team-Season Features from Detailed Results
# ============================================================
def build_advanced_team_features(reg_det: pd.DataFrame) -> pd.DataFrame:
    w = reg_det[[
        'Season','DayNum','WTeamID','WScore','LScore','WFGM','WFGA','WFGM3','WFGA3','WFTM','WFTA','WOR','WDR','WAst','WTO','WStl','WBlk',
        'LFGM','LFGA','LFGM3','LFGA3','LFTM','LFTA','LOR','LDR','LAst','LTO','LStl','LBlk'
    ]].copy()
    w.columns = [
        'Season','DayNum','TeamID','Pts','OppPts','FGM','FGA','FGM3','FGA3','FTM','FTA','OR','DR','Ast','TO','Stl','Blk',
        'OppFGM','OppFGA','OppFGM3','OppFGA3','OppFTM','OppFTA','OppOR','OppDR','OppAst','OppTO','OppStl','OppBlk'
    ]

    l = reg_det[[
        'Season','DayNum','LTeamID','LScore','WScore','LFGM','LFGA','LFGM3','LFGA3','LFTM','LFTA','LOR','LDR','LAst','LTO','LStl','LBlk',
        'WFGM','WFGA','WFGM3','WFGA3','WFTM','WFTA','WOR','WDR','WAst','WTO','WStl','WBlk'
    ]].copy()
    l.columns = [
        'Season','DayNum','TeamID','Pts','OppPts','FGM','FGA','FGM3','FGA3','FTM','FTA','OR','DR','Ast','TO','Stl','Blk',
        'OppFGM','OppFGA','OppFGM3','OppFGA3','OppFTM','OppFTA','OppOR','OppDR','OppAst','OppTO','OppStl','OppBlk'
    ]

    g = pd.concat([w, l], ignore_index=True)

    # Possession-based metrics
    g['Poss'] = g['FGA'] + 0.475 * g['FTA'] - g['OR'] + g['TO']
    g['OppPoss'] = g['OppFGA'] + 0.475 * g['OppFTA'] - g['OppOR'] + g['OppTO']

    # Four factors style rates
    g['eFGPct'] = safe_div(g['FGM'] + 0.5 * g['FGM3'], g['FGA'])
    g['TORate'] = safe_div(g['TO'], g['FGA'] + 0.44 * g['FTA'] + g['TO'])
    g['ORPct'] = safe_div(g['OR'], g['OR'] + g['OppDR'])
    g['FTRate'] = safe_div(g['FTM'], g['FGA'])

    g['Opp_eFGPct'] = safe_div(g['OppFGM'] + 0.5 * g['OppFGM3'], g['OppFGA'])
    g['Opp_TORate'] = safe_div(g['OppTO'], g['OppFGA'] + 0.44 * g['OppFTA'] + g['OppTO'])
    g['DRPct'] = safe_div(g['DR'], g['DR'] + g['OppOR'])

    g['OffRtg'] = 100 * safe_div(g['Pts'], g['Poss'])
    g['DefRtg'] = 100 * safe_div(g['OppPts'], g['OppPoss'])
    g['NetRtg'] = g['OffRtg'] - g['DefRtg']

    agg_cols = [
        'eFGPct', 'TORate', 'ORPct', 'FTRate',
        'Opp_eFGPct', 'Opp_TORate', 'DRPct',
        'OffRtg', 'DefRtg', 'NetRtg',
        'Ast', 'Stl', 'Blk', 'TO'
    ]

    out = g.groupby(['Season', 'TeamID'])[agg_cols].mean().reset_index()
    out.rename(columns={'Ast': 'AstPerGame', 'Stl': 'StlPerGame', 'Blk': 'BlkPerGame', 'TO': 'TOPerGame'}, inplace=True)
    return out

def build_massey_features(massey: pd.DataFrame, ranking_day: int = 133) -> pd.DataFrame:
    # Keep latest ranking up to tournament cutoff for each system/team/season
    m = massey[massey['RankingDayNum'] <= ranking_day].copy()
    m = m.sort_values(['Season', 'SystemName', 'TeamID', 'RankingDayNum'])
    m = m.groupby(['Season', 'SystemName', 'TeamID'], as_index=False).tail(1)

    agg = m.groupby(['Season', 'TeamID'])['OrdinalRank'].agg(
        MeanRank='mean', MedianRank='median', BestRank='min', WorstRank='max', RankStd='std', NumRankSystems='count'
    ).reset_index()

    key_systems = ['POM', 'SAG', 'MOR', 'DOL', 'COL']
    for sys_name in key_systems:
        tmp = m[m['SystemName'] == sys_name][['Season', 'TeamID', 'OrdinalRank']].rename(
            columns={'OrdinalRank': f'Rank_{sys_name}'}
        )
        agg = agg.merge(tmp, on=['Season', 'TeamID'], how='left')

    return agg

def build_conference_strength(team_conf: pd.DataFrame, basic: pd.DataFrame) -> pd.DataFrame:
    merged = team_conf.merge(basic, on=['Season', 'TeamID'], how='left')
    conf = merged.groupby(['Season', 'ConfAbbrev']).agg(
        ConfAvgWinPct=('WinPct', 'mean'),
        ConfAvgMargin=('AvgMargin', 'mean'),
        ConfAvgPPG=('PPG', 'mean'),
        ConfSize=('TeamID', 'count')
    ).reset_index()

    out = team_conf.merge(conf, on=['Season', 'ConfAbbrev'], how='left')
    return out[['Season', 'TeamID', 'ConfAvgWinPct', 'ConfAvgMargin', 'ConfAvgPPG', 'ConfSize']]

In [ ]:
# ============================================================
# 3.4 Assemble Men/Women Team Feature Tables
# ============================================================
def assemble_team_features(reg_compact, reg_det, seeds, team_conf, massey=None):
    basic = build_basic_team_features(reg_compact)
    recent = build_recent_form(reg_compact, n_games=10)
    sos = build_strength_of_schedule(reg_compact, basic)
    adv = build_advanced_team_features(reg_det)
    conf = build_conference_strength(team_conf, basic)

    seed_feat = seeds[['Season', 'TeamID', 'Seed']].copy()
    seed_feat['SeedNum'] = parse_seed_num(seed_feat['Seed'])
    seed_feat = seed_feat[['Season', 'TeamID', 'SeedNum']]

    feat = basic.merge(adv, on=['Season', 'TeamID'], how='left')
    feat = feat.merge(sos, on=['Season', 'TeamID'], how='left')
    feat = feat.merge(recent, on=['Season', 'TeamID'], how='left')
    feat = feat.merge(conf, on=['Season', 'TeamID'], how='left')
    feat = feat.merge(seed_feat, on=['Season', 'TeamID'], how='left')

    if massey is not None:
        rank_feat = build_massey_features(massey)
        feat = feat.merge(rank_feat, on=['Season', 'TeamID'], how='left')

    return feat

m_features = assemble_team_features(
    reg_compact=data['m_reg'],
    reg_det=data['m_reg_det'],
    seeds=data['m_seeds'],
    team_conf=data['m_team_conf'],
    massey=data['massey']
)

w_features = assemble_team_features(
    reg_compact=data['w_reg'],
    reg_det=data['w_reg_det'],
    seeds=data['w_seeds'],
    team_conf=data['w_team_conf'],
    massey=None
)

print('Men features:', m_features.shape)
print('Women features:', w_features.shape)
display(m_features.head(3))

In [ ]:
# ============================================================
# 4. Build Matchup-Level Training Data
# ============================================================
def build_matchups(tourney: pd.DataFrame, features: pd.DataFrame) -> pd.DataFrame:
    g = tourney[['Season', 'WTeamID', 'LTeamID']].copy()
    g['Team1'] = g[['WTeamID', 'LTeamID']].min(axis=1)
    g['Team2'] = g[['WTeamID', 'LTeamID']].max(axis=1)
    g['Target'] = (g['WTeamID'] == g['Team1']).astype(int)

    f = features.copy()
    num_cols = [c for c in f.columns if c not in ['Season', 'TeamID'] and pd.api.types.is_numeric_dtype(f[c])]

    t1 = f[['Season', 'TeamID'] + num_cols].copy()
    t2 = t1.copy()

    x = g[['Season', 'Team1', 'Team2', 'Target']].merge(
        t1, left_on=['Season', 'Team1'], right_on=['Season', 'TeamID'], how='left'
    ).drop(columns=['TeamID'])

    x = x.merge(
        t2, left_on=['Season', 'Team2'], right_on=['Season', 'TeamID'],
        how='left', suffixes=('_T1', '_T2')
    ).drop(columns=['TeamID'])

    for col in num_cols:
        c1, c2 = f'{col}_T1', f'{col}_T2'
        if c1 in x.columns and c2 in x.columns:
            x[f'Diff_{col}'] = x[c1] - x[c2]

    return x

m_train = build_matchups(data['m_tourney'], m_features)
w_train = build_matchups(data['w_tourney'], w_features)

print('Men matchups:', m_train.shape, '| target mean:', round(m_train['Target'].mean(), 4))
print('Women matchups:', w_train.shape, '| target mean:', round(w_train['Target'].mean(), 4))

## 5. Modeling Strategy

We use two complementary probability models:
- **Calibrated Logistic Regression**: stable linear baseline
- **Calibrated Histogram Gradient Boosting**: non-linear interactions

Final probability is a weighted blend optimized for Brier robustness.

In [ ]:
# ============================================================
# 5.1 Train / Validate by Season
# ============================================================
def get_feature_matrix(df: pd.DataFrame):
    diff_cols = [c for c in df.columns if c.startswith('Diff_')]
    x = df[diff_cols].copy()
    y = df['Target'].astype(int).copy()
    seasons = df['Season'].copy()
    return x, y, seasons, diff_cols

def evaluate_probabilities(y_true, p_pred, name='model'):
    p_pred = np.clip(np.asarray(p_pred), 1e-6, 1 - 1e-6)
    return {
        'Model': name,
        'Brier': brier_score_loss(y_true, p_pred),
        'LogLoss': log_loss(y_true, p_pred)
    }

def fit_model_bundle(train_df: pd.DataFrame, valid_seasons=VALID_SEASONS):
    x, y, seasons, diff_cols = get_feature_matrix(train_df)

    is_valid = seasons.isin(valid_seasons)
    x_tr, y_tr = x[~is_valid], y[~is_valid]
    x_va, y_va = x[is_valid], y[is_valid]

    print(f'Train rows: {len(x_tr):,} | Valid rows: {len(x_va):,}')

    # 1) Logistic baseline
    logreg_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(C=0.5, max_iter=5000, random_state=RANDOM_STATE))
    ])
    logreg_pipe.fit(x_tr, y_tr)
    p_log = logreg_pipe.predict_proba(x_va)[:, 1]

    # 2) Non-linear boosting + calibration
    hgb_base = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', HistGradientBoostingClassifier(
            learning_rate=0.03,
            max_depth=5,
            max_iter=700,
            min_samples_leaf=25,
            random_state=RANDOM_STATE
        ))
    ])
    hgb_cal = CalibratedClassifierCV(hgb_base, method='isotonic', cv=3)
    hgb_cal.fit(x_tr, y_tr)
    p_hgb = hgb_cal.predict_proba(x_va)[:, 1]

    # Weighted blend
    p_blend = 0.4 * p_log + 0.6 * p_hgb

    rows = [
        evaluate_probabilities(y_va, p_log, 'Logistic'),
        evaluate_probabilities(y_va, p_hgb, 'HistGB+IsoCal'),
        evaluate_probabilities(y_va, p_blend, 'Blend(0.4/0.6)')
    ]
    metrics = pd.DataFrame(rows).sort_values('Brier')

    # Refit on full data for inference
    logreg_full = clone(logreg_pipe).fit(x, y)
    hgb_full = clone(hgb_cal).fit(x, y)

    bundle = {
        'logreg': logreg_full,
        'hgb_cal': hgb_full,
        'diff_cols': diff_cols
    }

    return bundle, metrics

m_bundle, m_metrics = fit_model_bundle(m_train, VALID_SEASONS)
w_bundle, w_metrics = fit_model_bundle(w_train, VALID_SEASONS)

print('\nMen validation metrics')
display(m_metrics)
print('Women validation metrics')
display(w_metrics)

In [ ]:
# ============================================================
# 5.2 Quick Calibration Plot (Validation Window)
# ============================================================
def calibration_bins(y_true, p_pred, n_bins=10):
    df = pd.DataFrame({'y': y_true, 'p': p_pred}).dropna().copy()
    df['bin'] = pd.qcut(df['p'], q=n_bins, duplicates='drop')
    cal = df.groupby('bin').agg(pred=('p', 'mean'), obs=('y', 'mean'), n=('y', 'size')).reset_index(drop=True)
    return cal

def plot_calibration(train_df, bundle, title):
    x, y, seasons, _ = get_feature_matrix(train_df)
    mask = seasons.isin(VALID_SEASONS)
    x_va, y_va = x[mask], y[mask]

    p_log = bundle['logreg'].predict_proba(x_va)[:, 1]
    p_hgb = bundle['hgb_cal'].predict_proba(x_va)[:, 1]
    p_blend = 0.4 * p_log + 0.6 * p_hgb

    fig, ax = plt.subplots(figsize=(6, 6))
    for name, pred in [('Logistic', p_log), ('HistGB+Iso', p_hgb), ('Blend', p_blend)]:
        cal = calibration_bins(y_va, pred, n_bins=10)
        ax.plot(cal['pred'], cal['obs'], marker='o', label=name)

    ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
    ax.set_title(title)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Observed win rate')
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_calibration(m_train, m_bundle, 'Men: Calibration in Validation Seasons')
plot_calibration(w_train, w_bundle, 'Women: Calibration in Validation Seasons')

## 6. Inference Pipeline and Submission

Build matchup features for Stage 2 IDs, apply the trained men/women bundles, blend probabilities, and export a submission file.

In [ ]:
# ============================================================
# 6.1 Build Prediction Matchups from Submission IDs
# ============================================================
def build_prediction_matchups(sub_parsed: pd.DataFrame, features: pd.DataFrame) -> pd.DataFrame:
    num_cols = [c for c in features.columns if c not in ['Season', 'TeamID'] and pd.api.types.is_numeric_dtype(features[c])]

    f = features[['Season', 'TeamID'] + num_cols].copy()

    p = sub_parsed[['ID', 'Season', 'Team1', 'Team2']].copy()
    p = p.merge(f, left_on=['Season', 'Team1'], right_on=['Season', 'TeamID'], how='left').drop(columns=['TeamID'])
    p = p.merge(f, left_on=['Season', 'Team2'], right_on=['Season', 'TeamID'], how='left', suffixes=('_T1', '_T2')).drop(columns=['TeamID'])

    for col in num_cols:
        c1, c2 = f'{col}_T1', f'{col}_T2'
        if c1 in p.columns and c2 in p.columns:
            p[f'Diff_{col}'] = p[c1] - p[c2]

    return p

def predict_bundle(bundle, pred_df):
    x = pred_df.copy()

    # Keep training feature set only
    for col in bundle['diff_cols']:
        if col not in x.columns:
            x[col] = np.nan
    x = x[bundle['diff_cols']]

    p_log = bundle['logreg'].predict_proba(x)[:, 1]
    p_hgb = bundle['hgb_cal'].predict_proba(x)[:, 1]
    p = 0.4 * p_log + 0.6 * p_hgb
    return np.clip(p, 0.01, 0.99)

sub = parse_submission_ids(data['sub_stage2'])
m_sub = sub[sub['Gender'] == 'M'].copy()
w_sub = sub[sub['Gender'] == 'W'].copy()

m_pred_df = build_prediction_matchups(m_sub, m_features)
w_pred_df = build_prediction_matchups(w_sub, w_features)

m_sub['Pred'] = predict_bundle(m_bundle, m_pred_df)
w_sub['Pred'] = predict_bundle(w_bundle, w_pred_df)

submission = pd.concat([m_sub[['ID', 'Pred']], w_sub[['ID', 'Pred']]], ignore_index=True)
submission = submission.rename(columns={'Pred': 'Pred'})
submission = submission.sort_values('ID').reset_index(drop=True)

print('Final submission shape:', submission.shape)
display(submission.head())
display(submission['Pred'].describe())

In [ ]:
# ============================================================
# 6.2 Save Submission
# ============================================================
OUT_PATH = Path('submission_stage2_modeling.csv')
submission.to_csv(OUT_PATH, index=False)
print(f'✅ Submission file written: {OUT_PATH.resolve()}')

## 7. Next Iterations

High-impact improvements you can add next:
1. Season-wise rolling CV with fold-wise OOF blending weights
2. Separate model variants by tournament round / seed-gap regime
3. Explicit uncertainty features (rank dispersion, volatility in last-N games)
4. Include secondary-tourneys and prior-season carryover signals